# Lezione 8 — Attenzione e Transformer

In questa lezione costruisci, passo dopo passo e dal vivo, il meccanismo che sostiene i modelli linguistici attuali: l'attenzione (*attention*). Riprendi il problema lasciato aperto nella Lezione 6 con le reti neurali ricorrenti (RNN, *recurrent neural network*) e arrivi fino al componente centrale del Transformer.

**Obiettivi di apprendimento**

- Spiegare perché il collo di bottiglia delle RNN limita l'elaborazione di sequenze lunghe
- Mettere in relazione i concetti di query, chiave e valore con un'analogia quotidiana
- Implementare in PyTorch l'attenzione scalata per prodotto scalare usando solo `matmul` e `softmax`
- Visualizzare e interpretare una mappa di calore dei pesi di attenzione
- Collocare l'attenzione multi-testa come componente centrale del Transformer

## Il collo di bottiglia delle RNN

Ricorda la Lezione 6: una rete neurale ricorrente (RNN), o la sua variante più robusta, la LSTM (memoria a breve e lungo termine, *long short-term memory*), elabora una sequenza passo dopo passo e aggiorna uno stato nascosto di dimensione fissa. Quello stato deve riassumere l'intera frase, che abbia cinque parole o cinquanta. Più la sequenza è lunga, più informazione si perde per strada, quasi sempre quella comparsa all'inizio.

L'attenzione risolve il problema in un altro modo: invece di comprimere tutto in un unico vettore, permette al modello di consultare di nuovo la sequenza completa a ogni passo e decidere quali parti contano in quel momento.

## Query, chiave e valore

Immagina di cercare un libro in una biblioteca enorme. Hai in mente una domanda precisa: è la query (*query*). Scorri i dorsi dei libri, ciascuno con un'etichetta che ne riassume il contenuto: sono le chiavi (*keys*). Confronti la tua query con ogni chiave e decidi quanto si somigliano. Non porti a casa l'etichetta: porti il contenuto del libro più pertinente, cioè il suo valore (*value*).

Un meccanismo di attenzione fa esattamente questo con i vettori: confronta una query con un insieme di chiavi, ottiene per ciascuna un peso di pertinenza tramite la funzione softmax, e combina i valori pesati da quei pesi.

## Attenzione scalata per prodotto scalare da zero

Ora implementi la formula esatta:

$$\text{Attenzione}(Q, K, V) = \text{softmax}\left(\frac{QK^{\top}}{\sqrt{d_k}}\right) V$$

Generi $Q$, $K$ e $V$ come tre proiezioni lineari distinte di uno stesso input $X$. Calcoli i punteggi con `matmul` (l'operatore `@`), li scali per la radice di $d_k$ e applichi softmax per ottenere i pesi di attenzione, che poi pesano i valori. Lavori su una frase giocattolo, già tokenizzata a mano.

In [ ]:
import torch

torch.manual_seed(0)

# Frase de ejemplo, ya tokenizada en palabras (en un modelo real usarías un tokenizador real)
tokens = ["el", "gato", "negro", "duerme", "en", "el", "sofa"]
n = len(tokens)
d = 8      # dimensión del embedding, pequeña solo para que el ejemplo sea legible
d_k = 8    # dimensión de consultas y claves

# Embeddings de partida: en un modelo entrenado vendrían de una capa de embedding aprendida;
# aquí los generamos al azar para centrarnos en el mecanismo de atención
X = torch.randn(n, d)

# Proyecciones lineales aprendibles: transforman el mismo embedding en tres papeles distintos
W_q = torch.randn(d, d_k) * 0.5
W_k = torch.randn(d, d_k) * 0.5
W_v = torch.randn(d, d_k) * 0.5

Q = X @ W_q   # consultas: qué busca cada palabra
K = X @ W_k   # claves: qué ofrece cada palabra al ser comparada
V = X @ W_v   # valores: qué contenido entrega cada palabra si es elegida


def softmax_manual(z, dim=-1):
    # Restas el máximo antes de la exponencial por estabilidad numérica,
    # el mismo truco que viste con softmax en la capa de salida de clasificación
    z = z - z.max(dim=dim, keepdim=True).values
    exp_z = torch.exp(z)
    return exp_z / exp_z.sum(dim=dim, keepdim=True)


def atencion(Q, K, V):
    d_k = Q.shape[-1]
    puntuaciones = Q @ K.transpose(-2, -1)          # producto punto: consulta contra cada clave
    puntuaciones = puntuaciones / (d_k ** 0.5)       # escalado por raíz de d_k
    pesos = softmax_manual(puntuaciones, dim=-1)     # cada fila suma uno: es una distribución
    salida = pesos @ V                               # combinación ponderada de los valores
    return salida, pesos


salida, pesos_atencion = atencion(Q, K, V)

print("Forma de la salida:", salida.shape)
print("Cada fila de pesos suma:", pesos_atencion.sum(dim=-1))
print("Pesos de atención:\n", pesos_atencion)

## Visualizza dove guarda il modello

I pesi di attenzione formano una matrice di dimensione n×n: la parola che chiede (query) su un asse, la parola interrogata (chiave) sull'altro. Disegni questa matrice come mappa di calore con `imshow`: le tonalità più chiare indicano i pesi più alti. Poiché gli embedding di questo esempio sono casuali, non aspettarti un pattern linguistico reale, ma la meccanica pura del meccanismo.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
plt.imshow(pesos_atencion.detach().numpy(), cmap="viridis")
plt.xticks(range(n), tokens, rotation=45, ha="right")
plt.yticks(range(n), tokens)
plt.xlabel("Clave (palabra observada)")
plt.ylabel("Consulta (palabra que pregunta)")
plt.title("Pesos de atención escalada por producto punto")
plt.colorbar(label="Peso de atención")
plt.tight_layout()
plt.show()

## Dall'attenzione al Transformer

Una singola attenzione mescola in un unico schema di pesi ogni tipo di relazione: sintattiche, semantiche, di prossimità. Il Transformer esegue più attenzioni in parallelo, ciascuna con le proprie proiezioni $W_i^Q$, $W_i^K$, $W_i^V$, addestrate in modo indipendente: è l'attenzione multi-testa (*multi-head attention*). Concatena le uscite delle h teste e le riproietta alla dimensione originale con una matrice $W^O$ appresa.

$$\text{testa}_i = \text{Attenzione}(QW_i^Q, KW_i^K, VW_i^V)$$
$$\text{MultiTesta}(Q,K,V) = \text{Concat}(\text{testa}_1,\dots,\text{testa}_h)\,W^O$$

Ogni testa tende a specializzarsi: una guarda alle parole vicine, un'altra cattura la concordanza grammaticale a lunga distanza, un'altra segue i riferimenti pronominali. Il Transformer impila diversi blocchi di questo tipo, uno dopo l'altro, e rinuncia del tutto alla ricorrenza e alle convoluzioni. Oggi non lo implementi per intero: nella prossima lezione costruisci un blocco di Transformer completo, con attenzione multi-testa, normalizzazione e una rete feed-forward, appoggiandoti direttamente al codice che hai appena scritto.

## Riepilogo dei punti chiave

- Le RNN e le LSTM soffrono di un collo di bottiglia: comprimono un'intera sequenza in un unico stato nascosto di dimensione fissa, e perdono informazione nelle sequenze lunghe
- L'attenzione sostituisce quel riassunto unico con un vettore di contesto calcolato a ogni passo, come somma pesata dell'intero input
- Query, chiave e valore sono tre proiezioni lineari distinte dello stesso input: la chiave decide dove guardare, il valore è ciò che effettivamente leggi
- L'attenzione scalata per prodotto scalare si calcola come softmax(QKᵀ/√d_k)V; la scalatura evita che il softmax saturi con dimensioni grandi
- L'attenzione multi-testa esegue più attenzioni in parallelo e lascia che ogni testa si specializzi in un tipo diverso di relazione tra parole
- Il Transformer impila blocchi di autoattenzione (*self-attention*) e rinuncia del tutto alla ricorrenza

**Apri ora il notebook di pratica di accompagnamento, `../notebooks_practica/clase08_practica_atencion_transformers.ipynb`, e metti in pratica ciò che hai appena visto.**